# 7.17 — DETR

DETR (DEtection TRansformer) treats object detection as **set prediction**: the model emits a fixed number of object queries, each query either claims one ground-truth object or learns the special no-object label. The key math shift is one-to-one matching: instead of producing many duplicate boxes and cleaning them with NMS, DETR builds a cost matrix from class confidence and box geometry, solves the lowest-cost assignment, and trains only those matched pairs plus no-object slots.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build DETR one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is exposed, including IoU, matching costs, set assignment, no-object slots, and ranked AP. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

### 1. Fixed object queries make detection a set problem

Classic detectors often create many candidate boxes, then remove duplicates afterward. DETR instead asks for a **fixed set** of query outputs. If an image has two objects and the model emits five queries, two queries should match real objects and the other three should say no-object. The order of the queries is not meaningful; query 0 is not inherently “the first object.”

In [ ]:
queries_w = np.arange(5)
true_names_w = np.array(["dog", "bike"])
print("number of object queries:", len(queries_w))
print("ground-truth objects:", true_names_w.tolist())
print("extra slots:", len(queries_w) - len(true_names_w))
assert len(queries_w) - len(true_names_w) == 3

▶ What you'll see: DETR has five output slots even though only two objects are present.

In [ ]:
query_centers_w = np.array([[0.18, 0.28], [0.72, 0.62], [0.55, 0.22], [0.30, 0.75], [0.84, 0.35]])
object_centers_w = np.array([[0.20, 0.30], [0.75, 0.65]])
plt.figure(figsize=(4, 3.4))
plt.scatter(query_centers_w[:, 0], query_centers_w[:, 1], s=90, label="queries", color="steelblue")
plt.scatter(object_centers_w[:, 0], object_centers_w[:, 1], s=160, marker="*", label="objects", color="darkorange")
for i_w, xy_w in enumerate(query_centers_w):
    plt.text(xy_w[0] + .015, xy_w[1], f"q{i_w}")
plt.xlim(0, 1); plt.ylim(0, 1); plt.title("1: fixed query slots"); plt.legend(); plt.show()

▶ What you'll see: more blue query slots than orange objects; matching must decide which slots are responsible.

*Why it's done this way:* A fixed output size makes the neural network simple, but images contain a variable number of objects. DETR resolves that mismatch by interpreting outputs as an unordered set with a no-object option. The math must therefore be permutation-invariant: swapping two query rows should not change whether the prediction set is good.

### 2. IoU turns boxes into geometric cost

A class label alone is not enough; a query that says “dog” but puts the box in the wrong place should be expensive. DETR uses overlap through IoU. For boxes `[x1, y1, x2, y2]`, intersection area divided by union area is high when boxes cover the same pixels, and `1 - IoU` becomes a cost.

In [ ]:
def box_area_w(box_w):
    return max(0.0, box_w[2] - box_w[0]) * max(0.0, box_w[3] - box_w[1])

def iou_w(a_w, b_w):
    ix1_w, iy1_w = max(a_w[0], b_w[0]), max(a_w[1], b_w[1])
    ix2_w, iy2_w = min(a_w[2], b_w[2]), min(a_w[3], b_w[3])
    inter_w = box_area_w([ix1_w, iy1_w, ix2_w, iy2_w])
    union_w = box_area_w(a_w) + box_area_w(b_w) - inter_w
    return 0.0 if union_w == 0 else inter_w / union_w
box_a_w = np.array([0., 0., 3., 3.])
box_b_w = np.array([1., 1., 4., 4.])
print("IoU:", round(iou_w(box_a_w, box_b_w), 3))
print("IoU cost:", round(1 - iou_w(box_a_w, box_b_w), 3))
assert round(1 - iou_w(box_a_w, box_b_w), 3) == 0.714

▶ What you'll see: the two 3×3 boxes overlap in a 2×2 square, so `1 - IoU ≈ 0.714`.

In [ ]:
plt.figure(figsize=(4, 3.4))
for box_w, color_w, name_w in [(box_a_w, "steelblue", "A"), (box_b_w, "darkorange", "B")]:
    plt.gca().add_patch(plt.Rectangle((box_w[0], box_w[1]), box_w[2]-box_w[0], box_w[3]-box_w[1], fill=False, lw=2, ec=color_w, label=name_w))
plt.xlim(-.2, 4.4); plt.ylim(-.2, 4.4); plt.gca().set_aspect("equal")
plt.title("2: IoU measures box overlap"); plt.legend(); plt.show()

▶ What you'll see: only the central 2×2 region overlaps; the rest contributes to the union penalty.

*Why it's done this way:* Coordinate L1 distance and IoU answer different questions. L1 says whether corners are numerically close; IoU says whether the predicted region actually covers the target region. DETR combines them because detection needs both stable coordinate gradients and overlap-aware localization.

### 3. Matching cost combines class confidence and box geometry

For each target object and each query prediction, DETR builds a cost. A small cost means “this query is a good explanation for this target.” The lesson formula uses a class term `-log p(class)`, an L1 box term, and an IoU term. The assignment solver will see only this matrix, so the weights define what “best match” means.

In [ ]:
class_probs_w = np.array([[0.80, 0.10], [0.20, 0.75], [0.55, 0.30]])
pred_boxes_w = np.array([[0.10, 0.10, 0.40, 0.40], [0.58, 0.55, 0.88, 0.90], [0.35, 0.15, 0.65, 0.45]])
target_classes_w = np.array([0, 1])
target_boxes_w = np.array([[0.12, 0.12, 0.42, 0.42], [0.60, 0.58, 0.90, 0.88]])
print("prediction boxes shape:", pred_boxes_w.shape)
print("target boxes shape:", target_boxes_w.shape)

▶ What you'll see: three predictions compete for two targets.

In [ ]:
cost_w = np.zeros((len(target_classes_w), len(pred_boxes_w)))
for t_w in range(len(target_classes_w)):
    for q_w in range(len(pred_boxes_w)):
        cls_w = -np.log(class_probs_w[q_w, target_classes_w[t_w]])
        l1_w = np.sum(np.abs(target_boxes_w[t_w] - pred_boxes_w[q_w]))
        giou_like_w = 1 - iou_w(target_boxes_w[t_w], pred_boxes_w[q_w])
        cost_w[t_w, q_w] = cls_w + 2.0 * l1_w + 1.0 * giou_like_w
print(np.round(cost_w, 3))
assert cost_w.shape == (2, 3)

▶ What you'll see: a 2×3 matrix where each row is a target and each column is a query.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.imshow(cost_w, cmap="magma_r", aspect="auto")
plt.colorbar(label="lower is better")
plt.xticks(range(3), ["q0", "q1", "q2"]); plt.yticks(range(2), ["target0", "target1"])
plt.title("3: DETR matching cost matrix"); plt.show()

▶ What you'll see: the darkest/lowest cells show which query each target would prefer locally.

*Why it's done this way:* Matching is only as good as its cost. `-log p(c)` rewards the right semantic label, L1 rewards nearby coordinates, and `1 - IoU` rewards actual overlap. The weights matter because a confident class with a terrible box and a great box with the wrong class can otherwise tie.

### 4. One-to-one assignment removes duplicate responsibility

If every target simply chose its cheapest query, two targets could choose the same prediction. DETR instead solves a one-to-one assignment: each target gets a different query, and each foreground query is responsible for at most one object. For tiny notebooks we can solve this exactly by searching all assignments from scratch.

In [ ]:
def exact_assignment_w(C_w):
    n_t_w, n_q_w = C_w.shape
    best_cost_w, best_cols_w = np.inf, None
    def search_w(row_w, used_w, cols_w, total_w):
        nonlocal best_cost_w, best_cols_w
        if row_w == n_t_w:
            if total_w < best_cost_w:
                best_cost_w, best_cols_w = total_w, cols_w.copy()
            return
        for col_w in range(n_q_w):
            if col_w not in used_w:
                search_w(row_w + 1, used_w | {col_w}, cols_w + [col_w], total_w + C_w[row_w, col_w])
    search_w(0, set(), [], 0.0)
    return np.arange(n_t_w), np.array(best_cols_w), best_cost_w
rows_w, cols_w, total_w = exact_assignment_w(cost_w)
print("matched target rows:", rows_w)
print("matched query cols:", cols_w)
print("total cost:", round(total_w, 3))
assert len(set(cols_w.tolist())) == len(cols_w)

▶ What you'll see: target 0 and target 1 receive different query columns.

In [ ]:
tiny_cost_w = np.array([[0.3, 1.1], [0.7, 0.5]])
diag_w = tiny_cost_w[0, 0] + tiny_cost_w[1, 1]
cross_w = tiny_cost_w[0, 1] + tiny_cost_w[1, 0]
print("diagonal cost:", diag_w, "cross cost:", cross_w)
assert round(diag_w, 1) == 0.8 and round(cross_w, 1) == 1.8

▶ What you'll see: the diagonal assignment wins because `0.8 < 1.8`.

*Why it's done this way:* Set prediction needs a bookkeeping step that is itself permutation-invariant. Exact assignment compares whole matchings, not independent cells, so a query cannot duplicate two objects. This is the training-time replacement for the duplicate-cleanup habit that NMS performs at inference in older detectors.

### 5. Unmatched queries learn the no-object class

After foreground matching, unused queries are not ignored. They are trained to predict the no-object class. This is subtle but essential: without an explicit empty target, every extra query would be rewarded only when it hallucinated a foreground object.

In [ ]:
num_queries_w = 5
matched_queries_w = np.array([0, 1])
labels_w = np.full(num_queries_w, "no-object", dtype=object)
labels_w[matched_queries_w] = np.array(["dog", "bike"])
print("query labels:", labels_w.tolist())
print("no-object count:", int(np.sum(labels_w == "no-object")))
assert int(np.sum(labels_w == "no-object")) == 3

▶ What you'll see: three of five fixed query slots become no-object targets.

In [ ]:
no_obj_probs_w = np.array([0.05, 0.08, 0.82, 0.91, 0.76])
empty_loss_w = -np.log(no_obj_probs_w[labels_w == "no-object"])
print("no-object losses:", np.round(empty_loss_w, 3))
print("mean empty loss:", round(float(empty_loss_w.mean()), 3))
plt.figure(figsize=(4.5, 3))
plt.bar([f"q{i_w}" for i_w in range(num_queries_w)], no_obj_probs_w, color=np.where(labels_w == "no-object", "seagreen", "gray"))
plt.ylim(0, 1); plt.title("5: unmatched queries should be empty"); plt.ylabel("p(no-object)"); plt.show()

▶ What you'll see: the unmatched green bars are rewarded when their no-object probability is high.

*Why it's done this way:* DETR emits more slots than objects on purpose, so the loss must define what an unused slot should do. The no-object class turns “nothing here” into a supervised target, which discourages duplicate foreground predictions without needing NMS.

### 6. Decoder attention lets each query gather image evidence

DETR's transformer decoder gives each object query attention weights over image feature tokens. A query is not tied to an anchor box; it learns where to look. We can build one attention step with NumPy: scores are query-key dot products, softmax turns them into weights, and the weighted value sum becomes query evidence.

In [ ]:
def softmax_w(z_w):
    z_w = z_w - np.max(z_w)
    e_w = np.exp(z_w)
    return e_w / e_w.sum()
query_vec_w = np.array([1.0, 0.2])
keys_w = np.array([[1.0, 0.1], [0.8, 0.0], [0.1, 1.0], [0.0, 0.9]])
values_w = np.array([[2.0, 0.0], [1.5, 0.1], [0.0, 2.0], [0.1, 1.7]])
scores_w = keys_w @ query_vec_w / np.sqrt(2)
weights_w = softmax_w(scores_w)
print("attention weights:", np.round(weights_w, 3))
assert round(float(weights_w.sum()), 6) == 1.0

▶ What you'll see: weights sum to 1 and concentrate on tokens whose keys align with the query.

In [ ]:
attended_w = weights_w @ values_w
print("attended evidence vector:", np.round(attended_w, 3))
plt.figure(figsize=(4.5, 3))
plt.bar(["tok0", "tok1", "tok2", "tok3"], weights_w, color="slateblue")
plt.title("6: one query attends over image tokens"); plt.ylabel("attention weight"); plt.show()

▶ What you'll see: one or two tokens dominate the query's evidence vector.

*Why it's done this way:* Attention gives every query a differentiable way to collect global image information. Instead of hand-designing anchors, the query vector learns a pattern of “what to ask for,” and the softmax-weighted value sum provides the visual evidence used for class and box prediction.

### 7. AP still evaluates ranked foreground predictions

DETR changes training and decoding, but detector quality is still often reported by precision-recall and AP. After no-object filtering, foreground predictions are ranked by confidence; as recall grows, precision usually falls. A small rectangle-rule AP calculation keeps the evaluation idea concrete.

In [ ]:
precision_w = np.array([1.00, 0.75, 0.60])
recall_widths_w = np.array([0.33, 0.34, 0.33])
terms_w = precision_w * recall_widths_w
ap_w = float(terms_w.sum())
print("AP terms:", np.round(terms_w, 3))
print("AP:", round(ap_w, 3))
assert round(ap_w, 3) == 0.783

▶ What you'll see: `0.33 + 0.255 + 0.198 = 0.783`.

In [ ]:
recall_w = np.cumsum(recall_widths_w)
plt.figure(figsize=(4.5, 3))
plt.step(np.r_[0, recall_w], np.r_[precision_w[0], precision_w], where="post", color="darkorange")
plt.fill_between(np.r_[0, recall_w], np.r_[precision_w[0], precision_w], step="post", alpha=.25, color="darkorange")
plt.xlim(0, 1); plt.ylim(0, 1.05); plt.title("7: AP area under precision-recall"); plt.xlabel("recall"); plt.ylabel("precision"); plt.show()

▶ What you'll see: AP is the shaded area under a ranked precision-recall curve.

*Why it's done this way:* Matching defines how DETR learns, but AP defines how detections are judged after ranking. This separation matters: a model can have elegant set training and still fail evaluation if its foreground confidence ranking or localization quality is poor.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses
> small numbers, prints every intermediate with an inline `# ->` result, draws one picture,
> and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Fixed queries leave no-object slots

DETR emits a fixed set of query outputs. If there are fewer objects than queries, the leftovers must learn the no-object label.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)
t1_query_ids = np.arange(4)
print("query ids:", t1_query_ids)  # -> [0,1,2,3]
t1_objects = np.array(["mug", "plant"])
print("objects:", t1_objects.tolist())  # -> ['mug','plant']
t1_empty = len(t1_query_ids) - len(t1_objects)
print("no-object slots:", t1_empty)  # -> 2
assert t1_empty == 2

plt.figure(figsize=(4, 3))
plt.bar(["objects", "empty queries"], [len(t1_objects), t1_empty], color=["darkorange", "seagreen"])
plt.ylabel("count")
plt.title("Toy 1 · fixed output set")
plt.show()

▶ What you'll see: four queries can cover two objects only if two slots are explicitly empty.

### ✍️ Toy 2 · IoU becomes a box cost

`1 - IoU` is small for well-overlapping boxes and large when geometry is poor.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)
t2_box_a = np.array([0.0, 0.0, 2.0, 2.0])
t2_box_b = np.array([1.0, 0.0, 3.0, 2.0])
print("box A:", t2_box_a)  # -> [0,0,2,2]
print("box B:", t2_box_b)  # -> [1,0,3,2]
t2_inter_w = min(t2_box_a[2], t2_box_b[2]) - max(t2_box_a[0], t2_box_b[0])
t2_inter_h = min(t2_box_a[3], t2_box_b[3]) - max(t2_box_a[1], t2_box_b[1])
print("intersection width/height:", t2_inter_w, t2_inter_h)  # -> 1.0 2.0
t2_inter = t2_inter_w * t2_inter_h
print("intersection area:", t2_inter)  # -> 2.0
t2_area_a = (t2_box_a[2] - t2_box_a[0]) * (t2_box_a[3] - t2_box_a[1])
t2_area_b = (t2_box_b[2] - t2_box_b[0]) * (t2_box_b[3] - t2_box_b[1])
print("areas:", t2_area_a, t2_area_b)  # -> 4.0 4.0
t2_union = t2_area_a + t2_area_b - t2_inter
print("union area:", t2_union)  # -> 6.0
t2_iou = t2_inter / t2_union
t2_cost = 1 - t2_iou
print("IoU and cost:", round(float(t2_iou), 3), round(float(t2_cost), 3))  # -> 0.333 0.667
assert round(float(t2_cost), 3) == 0.667

plt.figure(figsize=(4, 3.5))
for t2_box, t2_color, t2_name in [(t2_box_a, "steelblue", "A"), (t2_box_b, "darkorange", "B")]:
    plt.gca().add_patch(plt.Rectangle((t2_box[0], t2_box[1]), t2_box[2] - t2_box[0], t2_box[3] - t2_box[1], fill=False, edgecolor=t2_color, linewidth=2, label=t2_name))
plt.xlim(-0.2, 3.2)
plt.ylim(2.3, -0.3)
plt.gca().set_aspect("equal")
plt.legend()
plt.title("Toy 2 · IoU overlap")
plt.show()

▶ What you'll see: the boxes share half of each box, but only one third of their union.

### ✍️ Toy 3 · Matching cost adds class, L1, and IoU terms

Each cost-matrix cell scores one possible target-query pair using semantics and geometry together.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)
t3_class_probs = np.array([[0.70, 0.20], [0.20, 0.80], [0.45, 0.40]])
t3_pred_boxes = np.array([[0.0, 0.0, 1.1, 1.0], [1.9, 2.0, 3.0, 3.1], [0.5, 0.5, 1.5, 1.5]])
t3_target_classes = np.array([0, 1])
t3_target_boxes = np.array([[0.0, 0.0, 1.0, 1.0], [2.0, 2.0, 3.0, 3.0]])
print("class probabilities:\n", t3_class_probs)  # -> query x class table
print("target classes:", t3_target_classes)  # -> [0,1]

def t3_iou(t3_a, t3_b):
    t3_ix1 = max(t3_a[0], t3_b[0])
    t3_iy1 = max(t3_a[1], t3_b[1])
    t3_ix2 = min(t3_a[2], t3_b[2])
    t3_iy2 = min(t3_a[3], t3_b[3])
    t3_inter = max(0.0, t3_ix2 - t3_ix1) * max(0.0, t3_iy2 - t3_iy1)
    t3_area_a = (t3_a[2] - t3_a[0]) * (t3_a[3] - t3_a[1])
    t3_area_b = (t3_b[2] - t3_b[0]) * (t3_b[3] - t3_b[1])
    t3_union = t3_area_a + t3_area_b - t3_inter
    return 0.0 if t3_union == 0 else t3_inter / t3_union

t3_class_cost = np.zeros((2, 3))
t3_l1_cost = np.zeros((2, 3))
t3_iou_cost = np.zeros((2, 3))
for t3_t in range(2):
    for t3_q in range(3):
        t3_class_cost[t3_t, t3_q] = -np.log(t3_class_probs[t3_q, t3_target_classes[t3_t]])
        t3_l1_cost[t3_t, t3_q] = np.abs(t3_pred_boxes[t3_q] - t3_target_boxes[t3_t]).sum()
        t3_iou_cost[t3_t, t3_q] = 1 - t3_iou(t3_pred_boxes[t3_q], t3_target_boxes[t3_t])
print("class cost:\n", np.round(t3_class_cost, 3))  # -> [[0.357,1.609,0.799],[1.609,0.223,0.916]]
print("L1 cost:\n", np.round(t3_l1_cost, 3))  # -> [[0.1,8.0,2.0],[7.9,0.2,6.0]]
print("IoU cost:\n", np.round(t3_iou_cost, 3))  # -> [[0.091,1.0,0.857],[1.0,0.174,1.0]]
t3_total_cost = t3_class_cost + t3_l1_cost + t3_iou_cost
print("total cost:\n", np.round(t3_total_cost, 3))  # -> [[0.548,10.609,3.656],[10.509,0.597,7.916]]
assert t3_total_cost.shape == (2, 3)
assert np.argmin(t3_total_cost, axis=1).tolist() == [0, 1]

plt.figure(figsize=(4.6, 3))
plt.imshow(t3_total_cost, cmap="magma_r", aspect="auto")
plt.colorbar(label="lower is better")
plt.xticks(range(3), ["q0", "q1", "q2"])
plt.yticks(range(2), ["target0", "target1"])
plt.title("Toy 3 · matching cost matrix")
plt.show()

▶ What you'll see: target 0 prefers query 0, while target 1 prefers query 1 after all terms are added.

### ✍️ Toy 4 · One-to-one assignment compares whole matchings

Matching cannot let two targets claim the same query. The best assignment is the cheapest valid pair of distinct columns.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)
t4_cost = np.array([[0.4, 0.3, 2.0], [0.5, 0.7, 0.8]])
print("cost matrix:\n", t4_cost)  # -> [[0.4,0.3,2.0],[0.5,0.7,0.8]]
t4_assignments = []
for t4_first in range(3):
    for t4_second in range(3):
        if t4_second != t4_first:
            t4_total = t4_cost[0, t4_first] + t4_cost[1, t4_second]
            t4_assignments.append((t4_first, t4_second, t4_total))
print("assignment totals:", [(int(t4_a), int(t4_b), round(float(t4_c), 2)) for t4_a, t4_b, t4_c in t4_assignments])  # -> includes (1,0,0.8)
t4_best = min(t4_assignments, key=lambda t4_item: t4_item[2])
print("best target->query columns:", t4_best[:2])  # -> (1,0)
print("best total cost:", round(float(t4_best[2]), 2))  # -> 0.8
assert t4_best[:2] == (1, 0)

plt.figure(figsize=(5, 3))
plt.bar([f"{t4_a}->{t4_b}" for t4_a, t4_b, _ in t4_assignments], [t4_c for _, _, t4_c in t4_assignments], color="slateblue")
plt.ylabel("total cost")
plt.title("Toy 4 · valid one-to-one assignments")
plt.xticks(rotation=30)
plt.show()

▶ What you'll see: the cross assignment `(target0→q1, target1→q0)` wins with total cost `0.8`.

### ✍️ Toy 5 · Unmatched queries learn no-object

After foreground matching, unused query slots receive an explicit no-object target and a no-object loss.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)
t5_num_queries = 4
t5_matched = np.array([1, 3])
print("matched query ids:", t5_matched)  # -> [1,3]
t5_labels = np.full(t5_num_queries, "no-object", dtype=object)
t5_labels[t5_matched] = np.array(["cat", "ball"])
print("query labels:", t5_labels.tolist())  # -> ['no-object','cat','no-object','ball']
t5_no_probs = np.array([0.80, 0.10, 0.60, 0.20])
print("p(no-object):", t5_no_probs)  # -> [0.8,0.1,0.6,0.2]
t5_empty_mask = t5_labels == "no-object"
print("empty mask:", t5_empty_mask.astype(int))  # -> [1,0,1,0]
t5_empty_losses = -np.log(t5_no_probs[t5_empty_mask])
print("empty-slot losses:", np.round(t5_empty_losses, 3))  # -> [0.223,0.511]
t5_empty_mean = float(t5_empty_losses.mean())
print("mean no-object loss:", round(t5_empty_mean, 3))  # -> 0.367
assert round(t5_empty_mean, 3) == 0.367

plt.figure(figsize=(4.5, 3))
plt.bar([f"q{t5_i}" for t5_i in range(t5_num_queries)], t5_no_probs, color=np.where(t5_empty_mask, "seagreen", "gray"))
plt.ylim(0, 1)
plt.ylabel("p(no-object)")
plt.title("Toy 5 · green slots are empty targets")
plt.show()

▶ What you'll see: queries 0 and 2 are trained to put high probability on no-object.

### ✍️ Toy 6 · Decoder attention gathers image evidence

A DETR query compares itself to image-token keys, softmaxes the scores, and averages value vectors.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)
t6_query = np.array([1.0, 0.5])
t6_keys = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
t6_values = np.array([[2.0, 0.0], [0.0, 1.0], [1.0, 2.0]])
print("query:", t6_query)  # -> [1,0.5]
t6_scores = t6_keys @ t6_query / np.sqrt(t6_query.size)
print("attention scores:", np.round(t6_scores, 3))  # -> [0.707,0.354,1.061]
t6_shifted = t6_scores - np.max(t6_scores)
print("shifted scores:", np.round(t6_shifted, 3))  # -> [-0.354,-0.707,0.0]
t6_weights = np.exp(t6_shifted) / np.exp(t6_shifted).sum()
print("attention weights:", np.round(t6_weights, 3))  # -> [0.32,0.225,0.456]
t6_attended = t6_weights @ t6_values
print("attended vector:", np.round(t6_attended, 3))  # -> [1.095,1.136]
assert round(float(t6_weights.sum()), 6) == 1.0
assert np.allclose(np.round(t6_attended, 3), [1.095, 1.136])

plt.figure(figsize=(4.5, 3))
plt.bar(["tok0", "tok1", "tok2"], t6_weights, color="slateblue")
plt.ylim(0, 1)
plt.ylabel("weight")
plt.title("Toy 6 · one query attends to tokens")
plt.show()

▶ What you'll see: token 2 gets the largest weight because its key aligns best with the query.

### ✍️ Toy 7 · AP ranks foreground detections

After no-object slots are filtered, detections are ranked by confidence and AP sums precision over recall jumps.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)
t7_scores = np.array([0.90, 0.70, 0.40, 0.30])
print("ranked scores:", t7_scores)  # -> [0.9,0.7,0.4,0.3]
t7_is_tp = np.array([1, 0, 1, 1])
print("true-positive flags:", t7_is_tp)  # -> [1,0,1,1]
t7_cum_tp = np.cumsum(t7_is_tp)
t7_cum_fp = np.cumsum(1 - t7_is_tp)
print("cumulative TP:", t7_cum_tp)  # -> [1,1,2,3]
print("cumulative FP:", t7_cum_fp)  # -> [0,1,1,1]
t7_precision = t7_cum_tp / (t7_cum_tp + t7_cum_fp)
t7_recall = t7_cum_tp / 3
print("precision:", np.round(t7_precision, 3))  # -> [1.0,0.5,0.667,0.75]
print("recall:", np.round(t7_recall, 3))  # -> [0.333,0.333,0.667,1.0]
t7_widths = np.diff(np.r_[0.0, t7_recall])
print("recall widths:", np.round(t7_widths, 3))  # -> [0.333,0.0,0.333,0.333]
t7_ap = float(np.sum(t7_precision * t7_widths))
print("AP:", round(t7_ap, 3))  # -> 0.806
assert round(t7_ap, 3) == 0.806

plt.figure(figsize=(4.6, 3))
plt.step(np.r_[0.0, t7_recall], np.r_[t7_precision[0], t7_precision], where="post", color="darkorange")
plt.fill_between(np.r_[0.0, t7_recall], np.r_[t7_precision[0], t7_precision], step="post", alpha=0.25, color="darkorange")
plt.ylim(0, 1.05)
plt.xlabel("recall")
plt.ylabel("precision")
plt.title("Toy 7 · ranked AP = 0.806")
plt.show()

▶ What you'll see: the false positive at rank 2 creates a zero-width recall step but lowers later precision.

## 🛠️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

def box_area(box):
    return max(0.0, box[2] - box[0]) * max(0.0, box[3] - box[1])

def iou(box_a, box_b):
    ix1, iy1 = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
    ix2, iy2 = min(box_a[2], box_b[2]), min(box_a[3], box_b[3])
    inter = box_area([ix1, iy1, ix2, iy2])
    union = box_area(box_a) + box_area(box_b) - inter
    return 0.0 if union == 0 else inter / union

def softmax(z):
    z = np.asarray(z, dtype=float)
    e = np.exp(z - np.max(z))
    return e / e.sum()

def exact_assignment(cost):
    cost = np.asarray(cost, dtype=float)
    n_rows, n_cols = cost.shape
    best_cost, best_cols = np.inf, None
    def search(row, used, cols, total):
        nonlocal best_cost, best_cols
        if row == n_rows:
            if total < best_cost:
                best_cost, best_cols = total, cols.copy()
            return
        for col in range(n_cols):
            if col not in used:
                search(row + 1, used | {col}, cols + [col], total + cost[row, col])
    search(0, set(), [], 0.0)
    return np.arange(n_rows), np.array(best_cols), best_cost

def draw_boxes(boxes, labels, colors, title):
    plt.figure(figsize=(4, 3.5))
    ax = plt.gca()
    for box, label, color in zip(boxes, labels, colors):
        ax.add_patch(plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], fill=False, lw=2, ec=color))
        ax.text(box[0], box[1], label, color=color, weight="bold")
    plt.xlim(0, 1); plt.ylim(0, 1); plt.gca().set_aspect("equal")
    plt.title(title); plt.show()

## 🟢 Basics (warm-up)

### Basic 1 — Count fixed queries versus objects

**Goal.** See why DETR needs no-object slots, because the model emits a fixed number of queries while each image has a variable number of objects. We build it in 2 steps.

In [ ]:
num_queries_b1 = 5
objects_b1 = np.array(["dog", "bike"])
empty_slots_b1 = num_queries_b1 - len(objects_b1)
print("queries:", num_queries_b1, "objects:", len(objects_b1), "empty slots:", empty_slots_b1)
assert empty_slots_b1 == 3

▶ What you'll see: five predictions must represent only two real objects.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["objects", "no-object slots"], [len(objects_b1), empty_slots_b1], color=["darkorange", "seagreen"])
plt.title("Basic 1: fixed DETR output set")
plt.ylabel("count")
plt.show()

▶ What you'll see: the no-object bar is larger than the foreground-object bar.

👀 Takeaway: DETR handles variable object counts by making extra fixed queries learn no-object.

### Basic 2 — Store boxes as corner coordinates

**Goal.** Represent bounding boxes as `[x1, y1, x2, y2]`, because IoU and L1 costs need explicit coordinates. We build it in 2 steps.

In [ ]:
box_b2 = np.array([0.10, 0.20, 0.50, 0.70])
width_b2 = box_b2[2] - box_b2[0]
height_b2 = box_b2[3] - box_b2[1]
area_b2 = width_b2 * height_b2
print("width:", width_b2, "height:", height_b2, "area:", round(area_b2, 3))
assert round(area_b2, 3) == 0.2

▶ What you'll see: normalized coordinates make area a fraction of the image.

In [ ]:
draw_boxes([box_b2], ["box"], ["steelblue"], "Basic 2: one normalized box")

▶ What you'll see: a single rectangle positioned inside a unit square.

👀 Takeaway: corner coordinates are simple enough for L1 distance and overlap calculations.

### Basic 3 — Compute intersection area

**Goal.** Find the overlap rectangle between two boxes, because IoU starts with intersection area. We build it in 2 steps.

In [ ]:
a_b3 = np.array([0.0, 0.0, 0.6, 0.6])
b_b3 = np.array([0.3, 0.2, 0.9, 0.8])
inter_box_b3 = np.array([max(a_b3[0], b_b3[0]), max(a_b3[1], b_b3[1]), min(a_b3[2], b_b3[2]), min(a_b3[3], b_b3[3])])
inter_area_b3 = box_area(inter_box_b3)
print("intersection box:", inter_box_b3)
print("intersection area:", round(inter_area_b3, 3))
assert round(inter_area_b3, 3) == 0.12

▶ What you'll see: the overlap has width 0.3 and height 0.4.

In [ ]:
draw_boxes([a_b3, b_b3, inter_box_b3], ["A", "B", "A∩B"], ["steelblue", "darkorange", "crimson"], "Basic 3: intersection rectangle")

▶ What you'll see: the red rectangle is exactly the part shared by both boxes.

👀 Takeaway: intersection uses max of left/top corners and min of right/bottom corners.

### Basic 4 — Compute IoU and IoU cost

**Goal.** Convert overlap into a unitless localization score, because DETR's box matching cost includes `1 - IoU`. We build it in 2 steps.

In [ ]:
a_b4 = np.array([0., 0., 3., 3.])
b_b4 = np.array([1., 1., 4., 4.])
iou_b4 = iou(a_b4, b_b4)
cost_b4 = 1 - iou_b4
print("IoU:", round(iou_b4, 3), "cost:", round(cost_b4, 3))
assert round(cost_b4, 3) == 0.714

▶ What you'll see: imperfect overlap creates a nonzero cost.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["IoU", "1-IoU"], [iou_b4, cost_b4], color=["seagreen", "crimson"])
plt.ylim(0, 1); plt.title("Basic 4: overlap score and cost"); plt.show()

▶ What you'll see: as IoU rises, the cost term would fall.

👀 Takeaway: IoU rewards covering the same region, while `1 - IoU` is the penalty used in matching.

### Basic 5 — Compute L1 box distance

**Goal.** Measure corner-coordinate error, because DETR combines L1 distance with IoU in the matching cost. We build it in 2 steps.

In [ ]:
target_b5 = np.array([0.10, 0.10, 0.40, 0.40])
pred_b5 = np.array([0.12, 0.08, 0.43, 0.38])
abs_terms_b5 = np.abs(target_b5 - pred_b5)
l1_b5 = float(abs_terms_b5.sum())
print("absolute coordinate errors:", abs_terms_b5)
print("L1 distance:", round(l1_b5, 3))
assert round(l1_b5, 3) == 0.09

▶ What you'll see: L1 adds four small coordinate errors.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["x1", "y1", "x2", "y2"], abs_terms_b5, color="purple")
plt.title("Basic 5: L1 coordinate terms"); plt.ylabel("absolute error"); plt.show()

▶ What you'll see: each corner coordinate contributes independently to L1 distance.

👀 Takeaway: L1 gives a stable coordinate-level penalty that complements overlap-based IoU.

### Basic 6 — Turn class probability into class cost

**Goal.** Use `-log p(class)` as a cost, because confident correct classes should be cheap and uncertain ones expensive. We build it in 2 steps.

In [ ]:
probs_b6 = np.array([0.80, 0.20, 0.05])
class_costs_b6 = -np.log(probs_b6)
print("probabilities:", probs_b6)
print("class costs:", np.round(class_costs_b6, 3))
assert round(float(class_costs_b6[0]), 3) == 0.223

▶ What you'll see: high probability 0.80 becomes a small cost around 0.223.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["0.80", "0.20", "0.05"], class_costs_b6, color="darkorange")
plt.title("Basic 6: -log probability cost"); plt.ylabel("cost"); plt.show()

▶ What you'll see: the cost rises sharply as probability approaches zero.

👀 Takeaway: negative log probability strongly punishes confident mistakes or low confidence in the true class.

### Basic 7 — Build one prediction-target matching cost

**Goal.** Add class, L1, and IoU terms for one pair, because each cost-matrix cell represents one possible match. We build it in 2 steps.

In [ ]:
p_true_b7 = 0.8
pred_box_b7 = np.array([0.12, 0.08, 0.43, 0.38])
target_box_b7 = np.array([0.10, 0.10, 0.40, 0.40])
class_term_b7 = -np.log(p_true_b7)
l1_term_b7 = np.sum(np.abs(pred_box_b7 - target_box_b7))
iou_term_b7 = 1 - iou(pred_box_b7, target_box_b7)
total_b7 = class_term_b7 + 2 * l1_term_b7 + iou_term_b7
print("terms:", round(class_term_b7, 3), round(l1_term_b7, 3), round(iou_term_b7, 3))
print("weighted total:", round(total_b7, 3))

▶ What you'll see: the total is a weighted sum of semantic and geometric penalties.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["class", "2×L1", "1-IoU"], [class_term_b7, 2*l1_term_b7, iou_term_b7], color=["slateblue", "teal", "crimson"])
plt.title("Basic 7: one matching-cost cell"); plt.ylabel("cost contribution"); plt.show()

▶ What you'll see: the largest bar shows which term dominates this pair.

👀 Takeaway: DETR's assignment behavior changes when cost weights change.

### Basic 8 — Compare two assignments by hand

**Goal.** See why matching compares whole permutations, because locally cheap choices must respect one-to-one responsibility. We build it in 2 steps.

In [ ]:
C_b8 = np.array([[0.3, 1.1], [0.7, 0.5]])
diag_b8 = C_b8[0, 0] + C_b8[1, 1]
cross_b8 = C_b8[0, 1] + C_b8[1, 0]
print("diagonal:", diag_b8, "cross:", cross_b8)
assert round(diag_b8, 1) == 0.8 and round(cross_b8, 1) == 1.8

▶ What you'll see: the diagonal permutation is cheaper.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["diag", "cross"], [diag_b8, cross_b8], color=["seagreen", "crimson"])
plt.title("Basic 8: assignment totals"); plt.ylabel("total cost"); plt.show()

▶ What you'll see: assignment is decided by total cost, not by a single cell alone.

👀 Takeaway: matching selects a lowest-cost permutation between targets and queries.

### Basic 9 — Mark unmatched queries as no-object

**Goal.** Create labels after matching, because every unmatched fixed query still needs a training target. We build it in 2 steps.

In [ ]:
num_queries_b9 = 5
matched_b9 = np.array([0, 3])
labels_b9 = np.full(num_queries_b9, "no-object", dtype=object)
labels_b9[matched_b9] = ["dog", "bike"]
print("labels:", labels_b9.tolist())
assert int(np.sum(labels_b9 == "no-object")) == 3

▶ What you'll see: unmatched queries become no-object labels.

In [ ]:
colors_b9 = np.where(labels_b9 == "no-object", "seagreen", "darkorange")
plt.figure(figsize=(4.5, 3))
plt.bar([f"q{i_b9}" for i_b9 in range(num_queries_b9)], np.ones(num_queries_b9), color=colors_b9)
plt.title("Basic 9: foreground vs no-object queries"); plt.yticks([]); plt.show()

▶ What you'll see: foreground slots and no-object slots are explicitly separated.

👀 Takeaway: no-object supervision is what tells extra queries to stay empty.

### Basic 10 — Compute a small AP number

**Goal.** Sum precision-recall rectangles, because DETR is still evaluated as a ranked detector. We build it in 2 steps.

In [ ]:
precision_b10 = np.array([1.0, 0.75, 0.60])
widths_b10 = np.array([0.33, 0.34, 0.33])
terms_b10 = precision_b10 * widths_b10
ap_b10 = float(terms_b10.sum())
print("terms:", np.round(terms_b10, 3), "AP:", round(ap_b10, 3))
assert round(ap_b10, 3) == 0.783

▶ What you'll see: the AP calculation matches the lesson number 0.783.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["slice1", "slice2", "slice3"], terms_b10, color="navy")
plt.title("Basic 10: AP area pieces"); plt.ylabel("precision × recall width"); plt.show()

▶ What you'll see: AP is the sum of the three bar heights.

👀 Takeaway: DETR changes prediction structure, but AP still scores ranked foreground detections.

## 🟡 Easy

### Easy 1 — Build a full cost matrix

**Goal.** Construct target-by-query costs from class probability and box geometry, because Hungarian matching receives a matrix rather than raw predictions. We build it in 3 steps.

In [ ]:
probs_e1 = np.array([[0.85, 0.10], [0.15, 0.80], [0.45, 0.35]])
preds_e1 = np.array([[0.10, 0.10, 0.40, 0.40], [0.58, 0.57, 0.90, 0.88], [0.35, 0.10, 0.65, 0.42]])
targets_e1 = np.array([[0.12, 0.12, 0.42, 0.42], [0.60, 0.58, 0.88, 0.90]])
classes_e1 = np.array([0, 1])
print("predictions:", preds_e1.shape[0], "targets:", targets_e1.shape[0])

▶ What you'll see: three queries compete for two target objects.

In [ ]:
cost_e1 = np.zeros((2, 3))
for t_e1 in range(2):
    for q_e1 in range(3):
        cls_e1 = -np.log(probs_e1[q_e1, classes_e1[t_e1]])
        l1_e1 = np.sum(np.abs(targets_e1[t_e1] - preds_e1[q_e1]))
        ov_e1 = 1 - iou(targets_e1[t_e1], preds_e1[q_e1])
        cost_e1[t_e1, q_e1] = cls_e1 + 2*l1_e1 + ov_e1
print(np.round(cost_e1, 3))
assert cost_e1.shape == (2, 3)

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.imshow(cost_e1, cmap="magma_r", aspect="auto")
plt.colorbar(label="cost")
plt.xticks(range(3), ["q0", "q1", "q2"]); plt.yticks(range(2), ["target0", "target1"])
plt.title("Easy 1: full matching-cost matrix"); plt.show()

▶ What you'll see: low-cost cells align the first target with q0 and the second target with q1.

👀 Takeaway: DETR's loss first translates predictions into an assignment cost matrix.

### Easy 2 — Solve the one-to-one assignment

**Goal.** Use a from-scratch exact assignment solver, because each target should be matched to a distinct query. We build it in 3 steps.

In [ ]:
cost_e2 = np.array([[0.50, 2.80, 1.40], [2.60, 0.40, 1.20]])
rows_e2, cols_e2, total_e2 = exact_assignment(cost_e2)
print("rows:", rows_e2, "cols:", cols_e2, "total:", round(total_e2, 3))
assert cols_e2.tolist() == [0, 1]

▶ What you'll see: target 0 uses q0 and target 1 uses q1.

In [ ]:
chosen_e2 = np.zeros_like(cost_e2)
chosen_e2[rows_e2, cols_e2] = 1
print("chosen mask:\n", chosen_e2.astype(int))

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.imshow(cost_e2, cmap="magma_r", aspect="auto")
plt.scatter(cols_e2, rows_e2, s=180, facecolors="none", edgecolors="cyan", linewidths=2)
plt.xticks(range(3), ["q0", "q1", "q2"]); plt.yticks(range(2), ["target0", "target1"])
plt.title("Easy 2: exact assignment result"); plt.colorbar(label="cost"); plt.show()

▶ What you'll see: cyan circles mark the two selected one-to-one matches.

👀 Takeaway: assignment prevents two targets from using the same query.

### Easy 3 — Add no-object loss for unmatched queries

**Goal.** Combine foreground losses with no-object losses, because DETR trains every query output. We build it in 3 steps.

In [ ]:
num_queries_e3 = 5
matched_cols_e3 = np.array([0, 1])
fg_losses_e3 = np.array([0.35, 0.42])
no_obj_probs_e3 = np.array([0.02, 0.04, 0.80, 0.90, 0.70])
unmatched_e3 = np.array([q_e3 for q_e3 in range(num_queries_e3) if q_e3 not in matched_cols_e3])
print("unmatched queries:", unmatched_e3)
assert unmatched_e3.tolist() == [2, 3, 4]

▶ What you'll see: three queries are not paired with foreground targets.

In [ ]:
empty_losses_e3 = -np.log(no_obj_probs_e3[unmatched_e3])
total_loss_e3 = float(fg_losses_e3.sum() + 0.2 * empty_losses_e3.sum())
print("foreground loss sum:", round(float(fg_losses_e3.sum()), 3))
print("weighted empty loss sum:", round(float(0.2 * empty_losses_e3.sum()), 3))
print("total:", round(total_loss_e3, 3))

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["foreground", "0.2×empty"], [fg_losses_e3.sum(), 0.2*empty_losses_e3.sum()], color=["darkorange", "seagreen"])
plt.title("Easy 3: every query contributes to loss"); plt.ylabel("loss"); plt.show()

▶ What you'll see: no-object losses matter, but can be down-weighted because there are many empty slots.

👀 Takeaway: unmatched queries are supervised as empty rather than ignored.

### Easy 4 — Show why NMS is not the training fix

**Goal.** Compare duplicate predictions before matching, because DETR's one-to-one loss teaches only one duplicate to own the target. We build it in 3 steps.

In [ ]:
target_e4 = np.array([0.20, 0.20, 0.60, 0.60])
dup_boxes_e4 = np.array([[0.19, 0.21, 0.61, 0.59], [0.22, 0.18, 0.58, 0.62], [0.65, 0.20, 0.90, 0.50]])
ious_e4 = np.array([iou(target_e4, b_e4) for b_e4 in dup_boxes_e4])
print("IoU with target:", np.round(ious_e4, 3))
assert int(np.argmax(ious_e4)) == 0

▶ What you'll see: q0 and q1 are both strong duplicates for the same object.

In [ ]:
single_target_cost_e4 = (1 - ious_e4)[None, :]
rows_e4, cols_e4, total_e4 = exact_assignment(single_target_cost_e4)
print("matched duplicate query:", int(cols_e4[0]))
print("unmatched duplicates:", [i_e4 for i_e4 in range(3) if i_e4 not in cols_e4])

In [ ]:
draw_boxes([target_e4] + list(dup_boxes_e4), ["target", "q0", "q1", "q2"], ["black", "seagreen", "orange", "crimson"], "Easy 4: duplicate predictions")

▶ What you'll see: only one of the overlapping predictions should receive foreground responsibility.

👀 Takeaway: DETR's one-to-one matching trains duplicates away instead of relying on NMS to hide them later.

### Easy 5 — One attention query over image tokens

**Goal.** Compute a decoder-style attention readout, because object queries gather evidence from image tokens before class and box heads. We build it in 3 steps.

In [ ]:
q_e5 = np.array([1.0, 0.2])
K_e5 = np.array([[1.0, 0.1], [0.7, 0.2], [0.1, 1.0], [0.0, 0.8]])
V_e5 = np.array([[2.0, 0.0], [1.4, 0.2], [0.1, 2.0], [0.2, 1.5]])
scores_e5 = K_e5 @ q_e5 / np.sqrt(2)
print("scores:", np.round(scores_e5, 3))

▶ What you'll see: tokens aligned with the query receive larger dot-product scores.

In [ ]:
weights_e5 = softmax(scores_e5)
context_e5 = weights_e5 @ V_e5
print("weights:", np.round(weights_e5, 3))
print("context:", np.round(context_e5, 3))
assert round(float(weights_e5.sum()), 6) == 1.0

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["tok0", "tok1", "tok2", "tok3"], weights_e5, color="slateblue")
plt.title("Easy 5: query attention weights"); plt.ylabel("softmax weight"); plt.show()

▶ What you'll see: the query reads mostly from the image tokens whose keys match it.

👀 Takeaway: object queries learn where to collect evidence; they are not fixed anchors.

## 🔴 Advanced

### Advanced 1 — Sweep matching weights

**Goal.** Change class and box weights, because the assignment depends on how semantic confidence trades off against geometry. We build it in 3 steps.

In [ ]:
class_cost_a1 = np.array([[0.20, 0.70], [0.70, 0.20]])
box_cost_a1 = np.array([[0.60, 0.10], [0.10, 0.60]])
weights_a1 = np.array([0.5, 1.0, 2.0, 4.0])
choices_a1 = []
print("class cost:\n", class_cost_a1)
print("box cost:\n", box_cost_a1)

▶ What you'll see: class and box terms prefer opposite assignments.

In [ ]:
for w_a1 in weights_a1:
    C_a1 = class_cost_a1 + w_a1 * box_cost_a1
    _, cols_a1, total_a1 = exact_assignment(C_a1)
    choices_a1.append(0 if cols_a1.tolist() == [0, 1] else 1)
    print("box weight", w_a1, "cols", cols_a1.tolist(), "total", round(total_a1, 3))

In [ ]:
plt.figure(figsize=(5, 3))
plt.step(weights_a1, choices_a1, where="mid", color="crimson")
plt.yticks([0, 1], ["class-like diag", "box-like cross"])
plt.xlabel("box-cost weight"); plt.title("Advanced 1: weights can flip matching"); plt.show()

▶ What you'll see: increasing the geometry weight can switch the chosen permutation.

👀 Takeaway: DETR's λ weights are modeling decisions, not cosmetic constants.

### Advanced 2 — Exact set loss for a mini batch item

**Goal.** Compute a complete DETR-style loss for one image, because matching is only the first step before summing pair losses and no-object losses. We build it in 4 steps.

In [ ]:
probs_a2 = np.array([[0.82, 0.10, 0.08], [0.15, 0.78, 0.07], [0.20, 0.10, 0.70], [0.25, 0.15, 0.60]])
boxes_a2 = np.array([[0.10, 0.10, 0.40, 0.40], [0.60, 0.58, 0.90, 0.88], [0.30, 0.30, 0.50, 0.50], [0.75, 0.10, 0.95, 0.35]])
t_classes_a2 = np.array([0, 1])
t_boxes_a2 = np.array([[0.12, 0.12, 0.42, 0.42], [0.58, 0.56, 0.88, 0.90]])
print("queries:", len(boxes_a2), "targets:", len(t_boxes_a2))

▶ What you'll see: four queries must cover two objects and two empty slots.

In [ ]:
C_a2 = np.zeros((2, 4))
for t_a2 in range(2):
    for q_a2 in range(4):
        C_a2[t_a2, q_a2] = -np.log(probs_a2[q_a2, t_classes_a2[t_a2]]) + 2*np.sum(np.abs(t_boxes_a2[t_a2]-boxes_a2[q_a2])) + (1-iou(t_boxes_a2[t_a2], boxes_a2[q_a2]))
rows_a2, cols_a2, match_cost_a2 = exact_assignment(C_a2)
print("matched cols:", cols_a2.tolist(), "matching cost:", round(match_cost_a2, 3))
assert len(set(cols_a2.tolist())) == 2

In [ ]:
fg_loss_a2 = 0.0
for r_a2, c_a2 in zip(rows_a2, cols_a2):
    fg_loss_a2 += -np.log(probs_a2[c_a2, t_classes_a2[r_a2]]) + np.sum(np.abs(t_boxes_a2[r_a2]-boxes_a2[c_a2])) + (1-iou(t_boxes_a2[r_a2], boxes_a2[c_a2]))
unmatched_a2 = np.array([q_a2 for q_a2 in range(4) if q_a2 not in cols_a2])
empty_loss_a2 = -np.log(probs_a2[unmatched_a2, 2]).sum()
total_a2 = fg_loss_a2 + 0.2 * empty_loss_a2
print("foreground:", round(fg_loss_a2, 3), "empty weighted:", round(0.2*empty_loss_a2, 3), "total:", round(total_a2, 3))

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["matched pairs", "no-object"], [fg_loss_a2, 0.2*empty_loss_a2], color=["darkorange", "seagreen"])
plt.title("Advanced 2: complete set loss pieces"); plt.ylabel("loss"); plt.show()

▶ What you'll see: the total loss contains both matched foreground pairs and unmatched empty slots.

👀 Takeaway: DETR training is assignment plus supervised losses for every query.

### Advanced 3 — Permutation invariance check

**Goal.** Shuffle query order and verify the same set quality, because DETR predictions are unordered. We build it in 3 steps.

In [ ]:
C_a3 = np.array([[0.4, 2.0, 1.5], [1.7, 0.3, 1.2]])
_, cols_a3, total_a3 = exact_assignment(C_a3)
perm_a3 = np.array([2, 0, 1])
C_perm_a3 = C_a3[:, perm_a3]
_, cols_perm_a3, total_perm_a3 = exact_assignment(C_perm_a3)
print("original cols:", cols_a3.tolist(), "cost:", total_a3)
print("permuted cols:", cols_perm_a3.tolist(), "cost:", total_perm_a3)
assert round(total_a3, 6) == round(total_perm_a3, 6)

▶ What you'll see: the query column numbers change, but the best total cost does not.

In [ ]:
chosen_original_a3 = np.zeros_like(C_a3); chosen_original_a3[np.arange(2), cols_a3] = 1
chosen_perm_a3 = np.zeros_like(C_perm_a3); chosen_perm_a3[np.arange(2), cols_perm_a3] = 1
print("original chosen mask:\n", chosen_original_a3.astype(int))
print("permuted chosen mask:\n", chosen_perm_a3.astype(int))

In [ ]:
fig, ax_a3 = plt.subplots(1, 2, figsize=(7, 3))
ax_a3[0].imshow(C_a3, cmap="magma_r", aspect="auto"); ax_a3[0].set_title("original order")
ax_a3[1].imshow(C_perm_a3, cmap="magma_r", aspect="auto"); ax_a3[1].set_title("shuffled queries")
plt.suptitle("Advanced 3: set loss ignores arbitrary query order"); plt.show()

▶ What you'll see: heatmap columns move, but the best assignment value is unchanged.

👀 Takeaway: DETR's matching loss is permutation-invariant over query slots.

### Advanced 4 — Decode foreground predictions without NMS

**Goal.** Filter by foreground probability after no-object training, because DETR inference expects duplicates to have been handled by one-to-one learning. We build it in 3 steps.

In [ ]:
class_probs_a4 = np.array([[0.80, 0.05, 0.15], [0.10, 0.76, 0.14], [0.20, 0.10, 0.70], [0.18, 0.12, 0.70]])
boxes_a4 = np.array([[0.10, 0.10, 0.42, 0.42], [0.60, 0.56, 0.88, 0.90], [0.12, 0.12, 0.40, 0.40], [0.70, 0.10, 0.90, 0.30]])
foreground_score_a4 = 1 - class_probs_a4[:, 2]
keep_a4 = foreground_score_a4 > 0.5
print("foreground scores:", np.round(foreground_score_a4, 3))
print("kept queries:", np.where(keep_a4)[0])
assert np.where(keep_a4)[0].tolist() == [0, 1]

▶ What you'll see: queries with high no-object probability are filtered out.

In [ ]:
kept_boxes_a4 = boxes_a4[keep_a4]
kept_labels_a4 = np.argmax(class_probs_a4[keep_a4, :2], axis=1)
print("kept labels:", kept_labels_a4.tolist())
print("kept boxes:\n", np.round(kept_boxes_a4, 2))

In [ ]:
draw_boxes(list(kept_boxes_a4), [f"class {x_a4}" for x_a4 in kept_labels_a4], ["seagreen", "darkorange"], "Advanced 4: DETR-style foreground decode")

▶ What you'll see: only two foreground boxes remain, with no NMS step in the code.

👀 Takeaway: DETR inference filters no-object predictions; duplicate suppression is intended to come from training.

### Advanced 5 — AP after confidence ranking

**Goal.** Rank detections, mark true positives by IoU, and compute precision-recall, because AP is still the detector score people report. We build it in 4 steps.

In [ ]:
gt_a5 = np.array([[0.10, 0.10, 0.40, 0.40], [0.60, 0.58, 0.90, 0.88]])
pred_a5 = np.array([[0.11, 0.10, 0.39, 0.41], [0.62, 0.58, 0.91, 0.90], [0.15, 0.15, 0.45, 0.45]])
scores_a5 = np.array([0.95, 0.80, 0.55])
order_a5 = np.argsort(scores_a5)[::-1]
print("rank order:", order_a5.tolist())

▶ What you'll see: detections are evaluated from highest confidence to lowest.

In [ ]:
used_a5 = np.zeros(len(gt_a5), dtype=bool)
tp_a5 = []
for idx_a5 in order_a5:
    overlaps_a5 = np.array([iou(pred_a5[idx_a5], g_a5) for g_a5 in gt_a5])
    best_a5 = int(np.argmax(overlaps_a5))
    is_tp_a5 = overlaps_a5[best_a5] >= 0.5 and not used_a5[best_a5]
    tp_a5.append(1 if is_tp_a5 else 0)
    if is_tp_a5:
        used_a5[best_a5] = True
print("TP flags:", tp_a5)
assert tp_a5 == [1, 1, 0]

In [ ]:
tp_cum_a5 = np.cumsum(tp_a5)
precision_a5 = tp_cum_a5 / (np.arange(len(tp_a5)) + 1)
recall_a5 = tp_cum_a5 / len(gt_a5)
print("precision:", np.round(precision_a5, 3))
print("recall:", np.round(recall_a5, 3))

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.step(np.r_[0, recall_a5], np.r_[1, precision_a5], where="post", color="navy")
plt.scatter(recall_a5, precision_a5, color="crimson")
plt.xlim(0, 1.05); plt.ylim(0, 1.05); plt.title("Advanced 5: ranked detection PR curve"); plt.xlabel("recall"); plt.ylabel("precision"); plt.show()

▶ What you'll see: the first two detections are true positives; the duplicate becomes a false positive.

👀 Takeaway: even NMS-free DETR must rank foreground boxes well to achieve high AP.